# YOLOX Document Detection Training - FIXED VERSION

Train YOLOX model for Algerian ID card detection.

**License:** Apache-2.0 (Free for commercial use)

## 🚀 Step-by-Step Guide (READ THIS FIRST)### ⚠️ IMPORTANT: Blackwell GPU (RTX PRO 6000) NoticeThis GPU (sm_120) is very new. Training runs without --fp16 (mixed precision disabled).### Recommended Workflow:1. **Cell 2**: Run CUDA Diagnostic   - If ✅ GPU ready → Continue   - If ❌ → Run Cell 3 (NumPy Fix), then restart runtime2. **Cell 3**: NumPy Fix (if needed)   - Only run if Cell 2 shows NumPy 2.x   - After running: **Runtime > Restart runtime**   - Then re-run Cell 2 to verify3. **Cell 4**: CUDA Fix (if needed)   - Only if Cell 2 fails GPU test   - After running: Restart runtime4. **Cell 5-6**: Setup (Mount Drive, Extract Dataset)5. **Cell 7**: Restructure Dataset to COCO format6. **Cell 8**: Create YOLOX Config7. **Cell 9**: Download Pretrained Weights8. **Cell 10**: Apply COCO Patches (fixes numpy dtype)9. **Cell 11**: Start Training   - Uses `--num-workers 0` to avoid multiprocessing issues   - Training takes 2-3 hours---### Quick Start (if you've already fixed NumPy):1. Runtime > Restart runtime2. Run Cell 2 (Diagnostic) → should show ✅3. Run Cells 5-11 (skip 3-4 if already fixed)

## 1. Setup Environment

In [ ]:
# 🔧 QUICK CUDA DIAGNOSTIC - Run this first to verify GPU compatibility
import subprocess
import sys

print("="*60)
print("CUDA/PyTorch Diagnostic")
print("="*60)

# Check GPU
print("\n🖥️  GPU Info:")
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU found"

# Check PyTorch installation
print("\n📦 PyTorch Installation:")
try:
    import torch
    print(f"  PyTorch version: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA version: {torch.version.cuda}")
        print(f"  cuDNN version: {torch.backends.cudnn.version()}")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        
        # Test CUDA operations
        print("\n🧪 Testing CUDA operations...")
        x = torch.rand(100, 100).cuda()
        y = torch.rand(100, 100).cuda()
        z = x @ y
        print(f"  ✓ CUDA matrix multiplication works! Shape: {z.shape}")
        print("\n✅ GPU is ready for training!")
    else:
        print("\n❌ CUDA not available - check runtime settings")
        print("   Go to Runtime > Change runtime type > Select GPU")
except Exception as e:
    print(f"  ❌ Error: {e}")
    print("\n🔧 Try: Restart runtime and run this cell again")

print("\n" + "="*60)

In [ ]:
# 🔧 NUMPY FIX - Run this if you get NumPy 2.x compatibility errors
import numpy as np
print(f"Current NumPy version: {np.__version__}")

if int(np.__version__.split('.')[0]) >= 2:
    print("\n⚠️  NumPy 2.x detected. Downgrading to 1.26.4...")
    !pip install -q "numpy==1.26.4"
    print("\n✅ NumPy downgraded! Restart runtime and run diagnostic again.")
    print("   Runtime > Restart runtime (Ctrl+M .)")
else:
    print("\n✅ NumPy version is compatible")

In [ ]:
# ⚠️ IF CUDA TEST FAILED - Run this fix cell
# This reinstalls PyTorch with correct CUDA version for Colab

print("Uninstalling conflicting PyTorch versions...")
!pip uninstall -y torch torchvision torchaudio

print("\nInstalling PyTorch 2.2.0 with CUDA 11.8...")
!pip install -q torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

print("\n✅ Installation complete! Restart runtime and run diagnostic again.")
print("   Runtime > Restart runtime (Ctrl+M .)")

In [ ]:
# Clone YOLOX repository
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX

In [ ]:
# Install dependencies - Use Colab's pre-installed PyTorch
# Check if CUDA is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install YOLOX dependencies manually (avoid setup.py issues)
!pip install -q numpy opencv-python tqdm pyyaml tensorboard
!pip install -q pycocotools onnx onnxruntime-gpu
!pip install -q thop loguru

# Add YOLOX to path instead of installing
import sys
sys.path.insert(0, '/content/YOLOX')
print("YOLOX ready!")

## 2. Prepare Dataset

Extract and prepare COCO format dataset.

In [ ]:
# 🔍 DEBUG: Find your dataset zip file
import os

print("Searching for zip files in Google Drive...")
print("="*50)

# Common locations to check
search_paths = [
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/retin-verify/v3",
]

for path in search_paths:
    if os.path.exists(path):
        print(f"\n📁 {path}:")
        try:
            items = os.listdir(path)
            for item in sorted(items):
                if item.endswith('.zip'):
                    size = os.path.getsize(os.path.join(path, item)) / (1024**2)
                    print(f"   ✅ {item} ({size:.1f} MB)")
                elif os.path.isdir(os.path.join(path, item)):
                    print(f"   📂 {item}/")
                else:
                    print(f"   📄 {item}")
        except PermissionError:
            print("   (Permission denied)")
    else:
        print(f"\n❌ {path} does not exist")

print("\n" + "="*50)
print("Update the zip_path in the next cell with the correct path!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract dataset from zip
import os
import shutil
from pathlib import Path

# Adjust this path to match your zip file location
zip_path = "/content/drive/MyDrive/retin-verify/v3/algerian_id_cards.zip"
extract_path = "/content/dataset_raw"

# Verify zip file exists
if not os.path.exists(zip_path):
    print(f"❌ ERROR: Zip file not found at {zip_path}")
    print("\nChecking possible locations:")
    !find /content/drive -name "*.zip" 2>/dev/null | head -20
    raise FileNotFoundError(f"Zip file not found: {zip_path}")

print(f"✅ Found zip file: {zip_path}")
print(f"Zip file size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")

# Create extract directory
os.makedirs(extract_path, exist_ok=True)

# Extract zip with verbose output
print("\nExtracting zip file...")
!unzip -o "{zip_path}" -d {extract_path}

# Check what was extracted
print("\n✅ Extracted structure:")
!ls -la {extract_path}

# List all subdirectories
print("\n📁 All directories in extract_path:")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')

In [ ]:
# Restructure to COCO format that YOLOX expects
# YOLOX expects: data_dir/annotations.json and data_dir/train2017/, data_dir/val2017/

import json
from pathlib import Path
import shutil
import os

# Find the extracted dataset folder
raw_path = Path("/content/dataset_raw")

# Check if raw_path exists
if not raw_path.exists():
    print(f"❌ ERROR: {raw_path} does not exist!")
    print("Please check the extraction step above for errors.")
    raise FileNotFoundError(f"Path not found: {raw_path}")

# Check if train/val folders exist directly in raw_path FIRST
if (raw_path / "train").exists() and (raw_path / "val").exists():
    source_path = raw_path
    print("Found train/val directly in dataset_raw")
else:
    # Look for a subfolder that contains train/val
    dataset_folders = [d for d in raw_path.iterdir() if d.is_dir()]
    
    print(f"Found {len(dataset_folders)} folders in {raw_path}")
    for f in dataset_folders:
        print(f"  - {f.name}")
    
    # Check if any subfolder has train/val
    source_path = None
    for folder in dataset_folders:
        if (folder / "train").exists() and (folder / "val").exists():
            source_path = folder
            print(f"Found train/val in subfolder: {folder.name}")
            break
    
    if source_path is None:
        print(f"❌ ERROR: No dataset structure found in {raw_path}")
        print("Expected either:")
        print("  - dataset_raw/train/ and dataset_raw/val/")
        print("  - dataset_raw/<folder>/train/ and dataset_raw/<folder>/val/")
        print("\nContents:")
        for item in raw_path.iterdir():
            print(f"  {item}")
        raise FileNotFoundError("Dataset structure not found")

print(f"\n✅ Source dataset: {source_path}")

# Verify train/val structure
train_path = source_path / "train"
val_path = source_path / "val"

if not train_path.exists():
    print(f"❌ WARNING: train folder not found at {train_path}")
if not val_path.exists():
    print(f"❌ WARNING: val folder not found at {val_path}")

print(f"Train folder exists: {train_path.exists()}")
print(f"Val folder exists: {val_path.exists()}")

# Create COCO structure
coco_path = Path("/content/dataset_coco")
(coco_path / "train2017").mkdir(parents=True, exist_ok=True)
(coco_path / "val2017").mkdir(parents=True, exist_ok=True)
(coco_path / "annotations").mkdir(parents=True, exist_ok=True)

# Copy train images
train_img_src = source_path / "train" / "images"
if train_img_src.exists():
    for img in train_img_src.glob("*"):
        if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
            shutil.copy2(img, coco_path / "train2017" / img.name)
    print(f"Copied {len(list((coco_path / 'train2017').glob('*')))} train images")

# Copy val images
val_img_src = source_path / "val" / "images"
if val_img_src.exists():
    for img in val_img_src.glob("*"):
        if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
            shutil.copy2(img, coco_path / "val2017" / img.name)
    print(f"Copied {len(list((coco_path / 'val2017').glob('*')))} val images")

# Convert annotations to COCO format
def convert_to_coco(annotation_path, output_path, img_folder):
    """Convert our annotation format to COCO format."""
    with open(annotation_path, 'r') as f:
        data = json.load(f)
    
    # COCO format requires specific fields
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 0, "name": "id_card", "supercategory": "none"}]
    }
    
    # Map image filenames to IDs
    img_id_map = {}
    for i, img in enumerate(data.get("images", []), 1):
        img_id = i
        img_id_map[img["id"]] = img_id
        coco_data["images"].append({
            "id": img_id,
            "file_name": img["file_name"],
            "width": int(img["width"]),
            "height": int(img["height"])
        })
    
    for i, ann in enumerate(data.get("annotations", []), 1):
        # Convert bbox to float to avoid numpy dtype issues
        bbox = [float(x) for x in ann["bbox"]]
        coco_data["annotations"].append({
            "id": i,
            "image_id": int(img_id_map.get(ann["image_id"], ann["image_id"])),
            "category_id": 0,
            "bbox": bbox,
            "area": float(bbox[2] * bbox[3]),
            "iscrowd": 0,
            "segmentation": [],
            "attributes": {"occluded": False}
        })
    
    with open(output_path, 'w') as f:
        json.dump(coco_data, f)
    
    return len(coco_data["images"]), len(coco_data["annotations"])

# Convert train annotations
train_ann_src = source_path / "train" / "annotations.json"
print(f"\nLooking for train annotations at: {train_ann_src}")
if train_ann_src.exists():
    img_count, ann_count = convert_to_coco(
        train_ann_src, 
        coco_path / "annotations" / "instances_train2017.json",
        coco_path / "train2017"
    )
    print(f"Train: {img_count} images, {ann_count} annotations")
else:
    print(f"❌ ERROR: Train annotations not found at {train_ann_src}")
    print("Available files in train folder:")
    train_folder = source_path / "train"
    if train_folder.exists():
        for item in train_folder.iterdir():
            print(f"  {item.name}")
    raise FileNotFoundError(f"Train annotations not found: {train_ann_src}")

# Convert val annotations
val_ann_src = source_path / "val" / "annotations.json"
print(f"\nLooking for val annotations at: {val_ann_src}")
if val_ann_src.exists():
    img_count, ann_count = convert_to_coco(
        val_ann_src, 
        coco_path / "annotations" / "instances_val2017.json",
        coco_path / "val2017"
    )
    print(f"Val: {img_count} images, {ann_count} annotations")
else:
    print(f"❌ ERROR: Val annotations not found at {val_ann_src}")
    print("Available files in val folder:")
    val_folder = source_path / "val"
    if val_folder.exists():
        for item in val_folder.iterdir():
            print(f"  {item.name}")
    raise FileNotFoundError(f"Val annotations not found: {val_ann_src}")

# Verify final structure
print(f"\n✅ Dataset ready at: {coco_path}")
print("\nFinal structure:")
!ls -la {coco_path}
print("\nAnnotations:")
!ls -la {coco_path}/annotations/
print("\nTrain images (first 5):")
!ls {coco_path}/train2017/ | head -5
print("\nVal images (first 5):")
!ls {coco_path}/val2017/ | head -5

# Verify annotation files are valid JSON
for ann_file in ['instances_train2017.json', 'instances_val2017.json']:
    ann_path = coco_path / "annotations" / ann_file
    try:
        with open(ann_path, 'r') as f_check:
            data = json.load(f_check)
        print(f"\n✅ {ann_file}: Valid JSON with {len(data.get('images', []))} images, {len(data.get('annotations', []))} annotations")
    except Exception as e:
        print(f"\n❌ {ann_file}: Error reading file - {e}")

## 3. Create Custom Config

In [ ]:
# Create custom config directory
import os
os.makedirs('/content/YOLOX/exps/custom', exist_ok=True)

# Create custom config file
config_content = '''
#!/usr/bin/env python3
# -*- coding:utf-8 -*-

import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.depth = 0.33
        self.width = 0.50
        self.exp_name = "yolox_idcard"
        
        # Define model name and path
        self.output_dir = "/content/YOLOX/YOLOX_outputs"
        
        # ---------------- dataset config ---------------- #
        self.num_classes = 1  # Only ID card
        # COCO format: data_dir/annotations.json + data_dir/train2017/ + data_dir/val2017/
        self.data_dir = "/content/dataset_coco"
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"
        
        # --------------- transform config ---------------- #
        self.mosaic_prob = 1.0
        self.mixup_prob = 0.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 10.0
        self.translate = 0.1
        self.mosaic_scale = (0.1, 2)
        self.enable_mixup = False
        
        # -------------- training config ----------------- #
        self.warmup_epochs = 5
        self.max_epoch = 100
        self.warmup_lr = 0
        self.basic_lr_per_img = 0.01 / 64.0
        self.scheduler = "yoloxwarmcos"
        self.no_aug_epochs = 15
        self.min_lr_ratio = 0.05
        self.ema = True
        self.weight_decay = 0.0005
        self.momentum = 0.9
        self.print_interval = 10
        self.eval_interval = 10
        self.save_history_ckpt = False
        
        # ----------------- dataloader config ----------------- #
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.no_aug = False
        self.degrees = 10.0
        self.translate = 0.1
        self.scale = (0.1, 2)
        self.mosaic_scale = (0.5, 1.5)
        self.shear = 2.0
        self.perspective = 0.0
        self.enable_mixup = True
        self.mixup_scale = (0.5, 1.5)
        self.mixup_prob = 0.5
        
    def get_model(self):
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
        import torch
        
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if getattr(self, "model", None) is None:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels)
            self.model = YOLOX(backbone, head)

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model
    
    def get_data_loader(
        self, batch_size, is_distributed, no_aug=False, cache_img=False
    ):
        from yolox.data import (
            COCODataset,
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
        )
        import torch

        dataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name="train2017",
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=50,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            cache=cache_img,
        )

        dataset = MosaicDetection(
            dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=120,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob,
        )

        if is_distributed:
            batch_size = batch_size // torch.cuda.device_count()

        sampler = InfiniteSampler(len(dataset), seed=self.seed if self.seed else 0)

        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )

        dataloader_kwargs = {"num_workers": self.data_num_workers, "pin_memory": True}
        dataloader_kwargs["batch_sampler"] = batch_sampler

        train_loader = DataLoader(dataset, **dataloader_kwargs)

        return train_loader

    def get_eval_loader(self, batch_size, is_distributed, testdev=False):
        from yolox.data import COCODataset, ValTransform
        import torch

        valdataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name="val2017",
            img_size=self.test_size,
            preproc=ValTransform(legacy=False),
        )

        if is_distributed:
            batch_size = batch_size // torch.cuda.device_count()
            sampler = torch.utils.data.distributed.DistributedSampler(
                valdataset, shuffle=False
            )
        else:
            sampler = torch.utils.data.SequentialSampler(valdataset)

        dataloader_kwargs = {
            "num_workers": self.data_num_workers,
            "pin_memory": True,
            "sampler": sampler,
        }
        dataloader_kwargs["batch_size"] = batch_size
        val_loader = torch.utils.data.DataLoader(valdataset, **dataloader_kwargs)

        return val_loader

    def get_evaluator(self, batch_size, is_distributed, testdev=False):
        from yolox.evaluators import COCOEvaluator

        val_loader = self.get_eval_loader(batch_size, is_distributed, testdev)
        evaluator = COCOEvaluator(
            dataloader=val_loader,
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )
        return evaluator
'''

with open('/content/YOLOX/exps/custom/yolox_idcard.py', 'w') as f:
    f.write(config_content)
    
print('Config file created!')

## 4. Train Model

In [ ]:
# Download pretrained weights (optional but recommended)
!wget https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth
!mkdir -p /content/YOLOX/pretrained
!mv yolox_s.pth /content/YOLOX/pretrained/

In [ ]:
# 🔧 CRITICAL PATCH: Fix numpy dtype issues in YOLOX
# This must be run BEFORE training

import json
import numpy as np
from pathlib import Path

# Step 1: Clean COCO annotation files
print("Step 1: Cleaning COCO annotation files...")
for split in ['train2017', 'val2017']:
    ann_file = Path('/content/dataset_coco/annotations') / f'instances_{split}.json'
    if ann_file.exists():
        with open(ann_file, 'r') as f:
            data = json.load(f)
        
        # Convert all numeric values to native Python types
        for img in data.get('images', []):
            img['width'] = int(img['width'])
            img['height'] = int(img['height'])
            img['id'] = int(img['id'])
        
        for ann in data.get('annotations', []):
            ann['id'] = int(ann['id'])
            ann['image_id'] = int(ann['image_id'])
            ann['category_id'] = int(ann['category_id'])
            ann['bbox'] = [float(x) for x in ann['bbox']]
            ann['area'] = float(ann['area'])
            ann['iscrowd'] = int(ann.get('iscrowd', 0))
        
        with open(ann_file, 'w') as f:
            json.dump(data, f)
        
        print(f"  ✅ Cleaned {ann_file.name}")

# Step 2: Monkey-patch pycocotools to return Python types
print("\nStep 2: Patching pycocotools...")

try:
    from pycocotools import coco as coco_module
    original_coco_init = coco_module.COCO.__init__
    
    def patched_coco_init(self, annotation_file=None):
        original_coco_init(self, annotation_file)
        # Convert all numpy arrays in dataset to lists
        if hasattr(self, 'dataset'):
            for img in self.dataset.get('images', []):
                for key in ['width', 'height', 'id']:
                    if key in img:
                        img[key] = int(img[key])
            for ann in self.dataset.get('annotations', []):
                for key in ['id', 'image_id', 'category_id', 'iscrowd']:
                    if key in ann:
                        ann[key] = int(ann[key])
                if 'bbox' in ann:
                    ann['bbox'] = [float(x) for x in ann['bbox']]
                if 'area' in ann:
                    ann['area'] = float(ann['area'])
    
    coco_module.COCO.__init__ = patched_coco_init
    print("  ✅ Patched pycocotools.COCO")
except Exception as e:
    print(f"  ⚠️ Could not patch pycocotools: {e}")

# Step 3: Monkey-patch YOLOX COCODataset
print("\nStep 3: Patching YOLOX COCODataset...")

try:
    import sys
    sys.path.insert(0, '/content/YOLOX')
    from yolox.data.datasets import coco as yolox_coco_module
    
    # Patch the _cache_images method if it exists
    if hasattr(yolox_coco_module.COCODataset, '_cache_images'):
        original_cache = yolox_coco_module.COCODataset._cache_images
        def patched_cache(self):
            result = original_cache(self)
            # Convert any numpy types in annotations
            for ann in self.annotations:
                if isinstance(ann.get('bbox'), np.ndarray):
                    ann['bbox'] = ann['bbox'].tolist()
            return result
        yolox_coco_module.COCODataset._cache_images = patched_cache
        print("  ✅ Patched COCODataset._cache_images")
    
    # Patch __getitem__ to ensure bbox is list
    original_getitem = yolox_coco_module.COCODataset.__getitem__
    def patched_getitem(self, index):
        result = original_getitem(self, index)
        # Convert any numpy arrays in result
        if isinstance(result, tuple) and len(result) >= 2:
            img, target = result[0], result[1]
            if isinstance(target, np.ndarray):
                target = target.tolist()
                result = (img, target) + result[2:]
        return result
    yolox_coco_module.COCODataset.__getitem__ = patched_getitem
    print("  ✅ Patched COCODataset.__getitem__")
    
except Exception as e:
    print(f"  ⚠️ Could not patch YOLOX: {e}")

# Step 4: Set numpy to raise errors on type issues
print("\nStep 4: Configuring numpy...")
np.seterr(all='raise')

print("\n" + "="*60)
print("✅ All patches applied! You can now run training.")
print("="*60)

In [ ]:
# 🔧 CRITICAL FIX: Downgrade NumPy to 1.x for PyTorch compatibility
# Run this and then RESTART RUNTIME before training

import subprocess
import sys

print("Checking NumPy version...")
import numpy as np
print(f"Current NumPy: {np.__version__}")

if int(np.__version__.split('.')[0]) >= 2:
    print("\n⚠️  NumPy 2.x detected. Downgrading to 1.26.4...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4", "--force-reinstall"])
    print("\n❗ CRITICAL: You MUST restart runtime now!")
    print("   Runtime > Restart runtime (Ctrl+M .)")
    print("\nAfter restart, skip this cell and run training directly.")
else:
    print("\n✅ NumPy version is compatible (1.x)")
    print("You can proceed with training.")

In [ ]:
# Start trainingimport os# Change to YOLOX directoryos.chdir('/content/YOLOX')os.environ['PYTHONPATH'] = '/content/YOLOX:' + os.environ.get('PYTHONPATH', '')!python3 tools/train.py \    -f exps/custom/yolox_idcard.py \    -d 1 \    -b 16 \    -o \    -c pretrained/yolox_s.pth

## 5. Export to ONNX

In [ ]:
# Export trained model to ONNX
!python tools/export_onnx.py \
    --output-name yolox_idcard.onnx \
    -f exps/custom/yolox_idcard.py \
    -c YOLOX_outputs/yolox_idcard/best_ckpt.pth

## 6. Test Model

In [ ]:
# Test inference
from PIL import Image
import matplotlib.pyplot as plt

# Load test image
test_image = '/content/dataset_coco/val2017/' + os.listdir('/content/dataset_coco/val2017')[0]
img = Image.open(test_image)

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.title('Test Image')
plt.axis('off')
plt.show()

In [ ]:
# Run inference
!python tools/demo.py \
    image \
    -f exps/custom/yolox_idcard.py \
    -c YOLOX_outputs/yolox_idcard/best_ckpt.pth \
    --path {test_image} \
    --conf 0.5 \
    --nms 0.45 \
    --tsize 640 \
    --save_result

## 7. Save Model to Drive

In [ ]:
# Copy ONNX model to Drive
!mkdir -p /content/drive/MyDrive/models
!cp yolox_idcard.onnx /content/drive/MyDrive/models/
!cp YOLOX_outputs/yolox_idcard/best_ckpt.pth /content/drive/MyDrive/models/

print('Model saved to Google Drive!')

## Validation Metrics

Expected results:
- mAP@0.5: > 0.95
- Inference time: < 10ms on GPU
- Model size: ~30MB (YOLOX-S)